In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv

# 노트북 실행 시 .env는 자동 로드되지 않아서 명시적으로 로드합니다.
# 보통은 노트북/프로젝트 루트에 .env가 있으므로 CWD 기준으로 시도합니다.
_dotenv_path = Path.cwd() / ".env"
if _dotenv_path.exists():
    load_dotenv(_dotenv_path)
else:
    load_dotenv()

from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.agents import AssistantAgent
# ai용 extension
from autogen_ext.models.openai import OpenAIChatCompletionClient
# rond-robin 종료 조건
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
# helper function, 실시간으로 발생하는 메세지를 출력
from autogen_agentchat.ui import Console

In [7]:
api_key = os.getenv("OPENAI_API_KEY") or os.getenv("LLAMA_API_KEY")
if not api_key:
    raise RuntimeError("OPENAI_API_KEY(또는 LLAMA_API_KEY)를 환경변수/.env에서 찾지 못했습니다.")

model = OpenAIChatCompletionClient(model="gpt-4o-mini", api_key=api_key)

# 첫번째 에이전트 , 명확성 에이전트 , 이메일의 모든 문장을 명확하고 읽기 쉽게 개선
clarity_agent = AssistantAgent(
    "ClarityAgent",
    model_client=model,
    system_message=
            """You are an expert editor focused on clarity and simplicity. 
            Your job is to eliminate ambiguity, redundancy, and make every sentence crisp and clear. 
            Don't worry about persuasion or tone — just make the message easy to read and understand.""",
)

# 두번째 에이전트 , 톤 에이전트 , 이메일의 어조와 전문성을 개선
tone_agent = AssistantAgent(
    "ToneAgent",
    model_client=model,
    system_message=
            """You are a communication coach focused on emotional tone and professionalism. 
            Your job is to make the email sound warm, confident, and human — while staying professional 
            and appropriate for the audience. Improve the emotional resonance, polish the phrasing, 
            and adjust any words that may come off as stiff, cold, or overly casual.""",
)

# 세번째 에이전트 , 설득 에이전트 , 이메일의 설득력을 개선 
## 논쟁을 구조화하고, 장점을 강조하고, 약하거나 소극적인 표현은 제거
persuasion_agent = AssistantAgent(
    "PersuasionAgent",
    model_client=model,
    system_message=
            """You are a persuasion expert trained in marketing, behavioral psychology, and copywriting. 
            Your job is to enhance the email's persuasive power: 
            improve call to action, structure arguments, and emphasize benefits. 
            Remove weak or passive language.""",
)

# 네번째 에이전트 , 합성 에이전트 , 모든 에이전트의 결과를 합치고 최종 결과물을 만들어 내는 에이전트
synthesizer_agent = AssistantAgent(
    "SynthesizerAgent",
    model_client=model,
    system_message=
            """You are an advanced email-writing specialist. Your role is to read all 
            prior agent responses and revisions, and then **synthesize the best ideas** into a unified, 
            polished draft of the email. Focus on: Integrating clarity, tone, and persuasion improvements; 
            Ensuring coherence, fluency, and a natural voice; Creating a version that feels professional, 
            effective, and readable.""",
)

# 다섯번째 에이전트 , 검토 에이전트 , 최종 결과물을 검토하고 최종 결과물을 만들어 내는 에이전트
## 이메일에 큰 결함이 있으먼 구체적인 개선안을 제공함.
## **이메일이 괜찮으면 새로운 줄에 `완료,대화종료` 라고 출력 (종료 트리거).
critic_agent = AssistantAgent(
    "CriticAgent",
    model_client=model,
    system_message=
            """You are an email quality evaluator. Your job is to perform a final review 
            of the synthesized email and determine if it meets professional standards. Review the email for: 
            Clarity and flow, appropriate professional tone, effective call-to-action, and overall coherence.
            Be constructive but decisive. 
            If the email has major flaws (unclear message, unprofessional tone, or missing key elements), provide ONE specific improvement suggestion. 
            If the email meets professional standards and communicates effectively, respond with 'The email meets professional standards.' 
            followed by `완료,대화종료` on a new line. 
            You should only approve emails that are perfect enough for professional use, dont settle.""",
)

In [8]:
text_termination = TextMentionTermination("완료,대화종료")
max_messages_termination = MaxMessageTermination(max_messages=30)

termination_condition = text_termination | max_messages_termination

In [9]:
team = RoundRobinGroupChat(
    participants=[
        clarity_agent,
        tone_agent,
        persuasion_agent,
        synthesizer_agent,
        critic_agent,
    ],
    termination_condition=termination_condition,
)

#team.run 으로 실행 시 모든 대화가 종료 후 한번에 출력됨
await Console(
    team.run_stream(
        task="안녕하세요 북중미 월드컵 대한민국대표팀 선수별 어울리는 상세한 전술을 제안드립니다"
    )
)

---------- TextMessage (user) ----------
안녕하세요 북중미 월드컵 대한민국대표팀 선수별 어울리는 상세한 전술을 제안드립니다
---------- TextMessage (ClarityAgent) ----------
안녕하세요. 북중미 월드컵에서 대한민국대표팀 선수별 어울리는 전술을 제안드립니다.
---------- TextMessage (ToneAgent) ----------
안녕하세요.

북중미 월드컵에서 대한민국 대표팀 각 선수에게 어울리는 전술에 대한 제안을 드리고자 합니다. 선수들의 장점을 최대한 발휘할 수 있는 전략을 함께 고민해보면 좋겠습니다. 

이와 관련하여 추가적인 논의가 필요하시다면 언제든지 말씀해 주세요. 감사합니다!
---------- TextMessage (PersuasionAgent) ----------
안녕하세요.

북중미 월드컵에서 대한민국 대표팀이 최고의 성과를 낼 수 있도록 각 선수에 맞춤형 전술을 제안드립니다. 선수들의 강점을 극대화하고, 팀의 전반적인 상승세를 이루어낼 수 있는 전략을 함께 구상해보면 좋겠습니다.

이 제안은 선수 개개인의 특성에 기반하여 팀워크와 경기력 향상을 도모하는 데 큰 도움이 될 것입니다. 추가적인 논의나 아이디어 공유가 필요하시면 언제든지 연락 주시기 바랍니다. 

확실한 성과를 위해 함께 노력해봅시다! 감사합니다!
---------- TextMessage (SynthesizerAgent) ----------
안녕하세요.

북중미 월드컵에서 대한민국 대표팀이 최고의 성과를 거둘 수 있도록, 각 선수에게 맞춤형으로 제안하는 전술에 대해 말씀드리고자 합니다. 선수들의 강점을 최대한 발휘하고 팀의 전반적인 성장을 이끌어내는 전략을 함께 고민해보면 좋겠습니다.

이 제안은 선수 개인의 특성을 기반으로 하여 팀워크와 경기력 향상에 큰 도움이 될 것입니다. 추가적인 의견이나 논의가 필요하시면 언제든지 말씀해 주시기 바랍니다.

성공을 위해 긴밀히 협력해 나갑시다. 감사합니다!
--

TaskResult(messages=[TextMessage(id='718fb88a-c32c-47a5-a933-72e70d0de69f', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 4, 1, 8, 55, 35, 803441, tzinfo=datetime.timezone.utc), content='안녕하세요 북중미 월드컵 대한민국대표팀 선수별 어울리는 상세한 전술을 제안드립니다', type='TextMessage'), TextMessage(id='94c932d8-5af8-462f-a746-47d0f0e040f8', source='ClarityAgent', models_usage=RequestUsage(prompt_tokens=85, completion_tokens=25), metadata={}, created_at=datetime.datetime(2026, 4, 1, 8, 55, 38, 816697, tzinfo=datetime.timezone.utc), content='안녕하세요. 북중미 월드컵에서 대한민국대표팀 선수별 어울리는 전술을 제안드립니다.', type='TextMessage'), TextMessage(id='5eb3a1b3-7bc3-48ff-a3a4-f2ba280c0dbc', source='ToneAgent', models_usage=RequestUsage(prompt_tokens=138, completion_tokens=73), metadata={}, created_at=datetime.datetime(2026, 4, 1, 8, 55, 41, 694152, tzinfo=datetime.timezone.utc), content='안녕하세요.\n\n북중미 월드컵에서 대한민국 대표팀 각 선수에게 어울리는 전술에 대한 제안을 드리고자 합니다. 선수들의 장점을 최대한 발휘할 수 있는 전략을 함께 고민해보면 좋겠습니다. \n\n이와 관련하여 추가적인 논의가 